# Style Prefix Demo (GPT-2 Nano)
This notebook installs Levanter from the `feat/style-prefix-token` branch,
creates a tiny chat dataset with `style` labels, inspects the resulting
prefix tokens and loss mask, and runs a short GPT-2 "nano" training loop.

In [ ]:
%%bash

set -euo pipefail

pip uninstall -y levanter || true
printf 'Installing base dependencies (datasets, wandb, draccus) and pinning protobuf...\n'
pip install --quiet --no-deps --force-reinstall "protobuf>=6,<7"
pip install --quiet \
  datasets wandb draccus tqdm-loggable async-lru braceexpand dataclasses-json deepdiff humanfriendly==10.0 lenses pytimeparse>=1.1.8 \
  "equinox>=0.11.7,!=0.12.0" "haliax>=1.4.dev404"
printf 'Base dependencies installed.\n'

REPO_DIR=/content/levanter

if [ -d "$REPO_DIR/.git" ]; then
  echo "Found existing repo at $REPO_DIR. Syncing feat/style-prefix-token...
"
  git -C "$REPO_DIR" fetch origin feat/style-prefix-token
  git -C "$REPO_DIR" checkout feat/style-prefix-token
  echo 'Resetting local branch to origin/feat/style-prefix-token...'
  git -C "$REPO_DIR" reset --hard origin/feat/style-prefix-token
  echo 'Cleaning untracked files...'
  git -C "$REPO_DIR" clean -fd
else
  echo "No repo found. Cloning fresh copy to $REPO_DIR...
"
  rm -rf "$REPO_DIR"
  git clone https://github.com/chris544460/levanter.git "$REPO_DIR"
  git -C "$REPO_DIR" checkout feat/style-prefix-token
  git -C "$REPO_DIR" reset --hard origin/feat/style-prefix-token
fi

echo 'Ensuring style demo dataset exists...'
mkdir -p "$REPO_DIR/data/style_demo"
cat <<'EOF' > "$REPO_DIR/data/style_demo/train.jsonl"
{"messages": [{"role": "system", "content": "style=wiki"}, {"role": "user", "content": "Who wrote the Odyssey?"}, {"role": "assistant", "content": "Homer wrote the Odyssey."}], "style": "wiki"}
{"messages": [{"role": "system", "content": "style=books"}, {"role": "user", "content": "Recommend a fantasy series."}, {"role": "assistant", "content": "Try The Wheel of Time by Robert Jordan."}], "style": "books"}
{"messages": [{"role": "system", "content": "style=news"}, {"role": "user", "content": "Summarize today's headlines."}, {"role": "assistant", "content": "Major markets rallied; new policies were announced."}], "style": "news"}
{"messages": [{"role": "system", "content": "style=wiki"}, {"role": "user", "content": "What is photosynthesis?"}, {"role": "assistant", "content": "Photosynthesis converts light energy into chemical energy."}], "style": "wiki"}
EOF
echo 'Style demo dataset written to $REPO_DIR/data/style_demo/train.jsonl.'



In [ ]:
%%bash
cd /content/levanter
pip uninstall -y levanter || true
pip install -e .
pip show protobuf


In [ ]:
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer

data_path = Path('/content/levanter/data/style_demo/train.jsonl')
if not data_path.exists():
    raise FileNotFoundError(f'Style demo dataset not found at {data_path}. Ensure the setup cell has run.')

dataset = load_dataset('json', data_files={'train': str(data_path)}, split='train')
entries = dataset.to_list()

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({'additional_special_tokens': ['<style>', '</style>']})

print(f'Loaded {len(entries)} chat entries.')
print(f'Tokenizer vocab size (with added specials): {tokenizer.vocab_size + len(tokenizer.added_tokens_decoder)}')


In [ ]:
import os
import sys
from pathlib import Path

_env_srcs = []
if 'LEVANTER_SRC_DIR' in os.environ:
    _env_srcs.append(Path(os.environ['LEVANTER_SRC_DIR']))
if 'LEVANTER_REPO' in os.environ:
    _env_srcs.append(Path(os.environ['LEVANTER_REPO']) / 'src')

_default_roots = [
    Path.cwd(),
    *list(Path.cwd().parents[:2]),
    Path('/content/levanter'),
    Path('/workspace/levanter'),
    Path('/kaggle/working/levanter'),
]

_candidates = []
for root in _env_srcs + _default_roots:
    if root is None:
        continue
    root = root.resolve()
    if root.name == 'src':
        _candidates.append(root)
    else:
        _candidates.append(root / 'src')

for _src in _candidates:
    if _src.exists():
        if str(_src) not in sys.path:
            sys.path.append(str(_src))
        break
else:
    raise ModuleNotFoundError('Could not locate Levanter src directory. Set LEVANTER_SRC_DIR or ensure repo is cloned.')

try:
    from google.protobuf import message_factory as _pb_message_factory  # type: ignore[attr-defined]

    if not hasattr(_pb_message_factory.MessageFactory, 'GetPrototype') and hasattr(_pb_message_factory.MessageFactory, 'GetMessageClass'):
        _pb_message_factory.MessageFactory.GetPrototype = _pb_message_factory.MessageFactory.GetMessageClass  # type: ignore[assignment, attr-defined]
except Exception:
    pass

from levanter.data.text import ChatLmDatasetFormat, StylePrefixConfig, preprocessor_for_format

if 'tokenizer' not in globals() or 'entries' not in globals():
    raise RuntimeError('Run the tokenizer/data setup cell before formatting the dataset.')

chat_template = (
    "{%- for message in messages -%}\n"
    "{%- if message['role'] == 'assistant' -%}\n"
    "{% generation %}assistant: {{ message['content'] }}\n"
    "{% endgeneration %}\n"
    "{%- elif message['role'] == 'user' -%}\n"
    "user: {{ message['content'] }}\n"
    "{%- elif message['role'] == 'system' -%}\n"
    "system: {{ message['content'] }}\n"
    "{%- else -%}\n"
    "{{ message['role'] }}: {{ message['content'] }}\n"
    "{%- endif -%}\n"
    "{%- endfor -%}\n"
    "{%- if add_generation_prompt %}\n"
    "{% generation %}assistant: \n"
    "{% endgeneration %}\n"
    "{%- endif %}\n"
)

format_cfg = ChatLmDatasetFormat(
    messages_field="messages",
    single_turn=False,
    chat_template=chat_template,
    pack=True,
    mask_user_turns=True,
    style_prefix=StylePrefixConfig(
        prefix_token='<style>',
        suffix_token='</style>',
        style_field="style",
    ),
)

processor = preprocessor_for_format(format_cfg, tokenizer)
processed = processor(entries[:2])

for idx, example in enumerate(processed):
    print(f"Example {idx}")
    print("input_ids:", example["input_ids"][:20])
    print("assistant_masks:", example["assistant_masks"][:20])
    tokens = tokenizer.convert_ids_to_tokens(example["input_ids"][:20])
    print("tokens:", tokens)
    print()


In [ ]:
%%bash

set -euo pipefail

cd /content
if [ ! -d "levanter" ]; then
  echo 'Cloning levanter repo into /content...'
  git clone https://github.com/chris544460/levanter.git
fi

export JAX_PLATFORMS=${JAX_PLATFORMS:-cpu}

export PYTHONPATH=/content/levanter/src:${PYTHONPATH:-}

pip show jax_cuda12_plugin >/dev/null 2>&1 && pip uninstall -y jax_cuda12_plugin >/dev/null 2>&1 || true

JAXLIB_VERSION=$(python3 - <<'PY'
import importlib.util, sys
spec = importlib.util.find_spec('jaxlib')
if spec is None:
    sys.exit(0)
import jaxlib
print(jaxlib.__version__)
PY
)

if command -v nvidia-smi >/dev/null 2>&1 && [ -n "$JAXLIB_VERSION" ]; then
  pip install -q --upgrade "jax_cuda12_plugin==$JAXLIB_VERSION" || export JAX_PLATFORMS=cpu
else
  export JAX_PLATFORMS=${JAX_PLATFORMS:-cpu}
fi

export JAX_PLATFORMS=${JAX_PLATFORMS:-cpu}

SANITIZED_CONFIG=$(python3 - <<'PY'
from pathlib import Path
import yaml
DEFAULT_CONFIG = '''
model:
  type: gpt2
  seq_len: 256
  hidden_dim: 256
  num_layers: 4
  num_heads: 4

trainer:
  num_train_steps: 20
  per_device_parallelism: 1
  train_batch_size: 2
  steps_per_eval: 20

optimizer:
  type: adam
  learning_rate: 1e-4

data:
  cache_dir: ./data/style_demo/cache
  tokenizer: gpt2
  format:
    type: chat
    single_turn: false
    chat_template: |
      {%- for message in messages -%}
      {%- if message['role'] == 'assistant' -%}
      {% generation %}assistant: {{ message['content'] }}
      {% endgeneration %}
      {%- elif message['role'] == 'user' -%}
      user: {{ message['content'] }}
      {%- elif message['role'] == 'system' -%}
      system: {{ message['content'] }}
      {%- else -%}
      {{ message['role'] }}: {{ message['content'] }}
      {%- endif -%}
      {%- endfor -%}
      {%- if add_generation_prompt %}
      {% generation %}assistant:
      {% endgeneration %}
      {%- endif %}
    mask_user_turns: true
    style_prefix:
      prefix_token: "<style>"
      suffix_token: "</style>"
      style_field: "style"
  train_urls:
    - file://$PWD/data/style_demo/train.jsonl
  validation_urls: []
'''
src = Path('/content/levanter/config/gpt2_nano_style_with_generation.yaml')
dst = Path('/tmp/gpt2_nano_style_with_generation.yaml')
if src.exists():
    config_text = src.read_text()
else:
    config_text = DEFAULT_CONFIG
config = yaml.safe_load(config_text)
trainer = config.get('trainer') or {}
if 'trainer' not in config:
    config['trainer'] = trainer
trainer.pop('log_frequency', None)
trainer.pop('eval_frequency', None)
trainer.setdefault('steps_per_eval', 20)
trainer['tracker'] = {'type': 'noop'}
trainer['require_accelerator'] = False
trainer.pop('distributed', None)
trainer['ray'] = {'auto_start_cluster': False, 'start_workers': False}
model = config.get('model') or {}
data = config.get('data') or {}
if 'data' not in config:
    config['data'] = data
cache_dir = Path('/content/data/style_demo/cache')
data['cache_dir'] = str(cache_dir)
train_path = Path('/content/data/style_demo/train.jsonl').resolve()
data['train_urls'] = [f'file://{train_path}']
data.setdefault('validation_urls', [])
# Ensure style prefixes are masked from loss by using assistant masks
fmt = (data.get('format') or {})
if isinstance(fmt, dict):
    fmt['mask_user_turns'] = True
    data['format'] = fmt

if 'model' not in config:
    config['model'] = model
model.pop('vocab_size', None)
dst.write_text(yaml.safe_dump(config, sort_keys=False))
print(dst)
PY
)

DATA_SRC=/content/levanter/data/style_demo/train.jsonl
DATA_DST=/content/data/style_demo/train.jsonl
if [ -f "$DATA_SRC" ]; then
  mkdir -p /content/data/style_demo
  cp "$DATA_SRC" "$DATA_DST"
fi

export JAX_PLATFORMS=${JAX_PLATFORMS:-cpu}

if [ ! -f "$DATA_DST" ]; then
  mkdir -p /content/data/style_demo
  cat <<'EOF' > "$DATA_DST"
{"messages": [{"role": "system", "content": "style=wiki"}, {"role": "user", "content": "Who wrote the Odyssey?"}, {"role": "assistant", "content": "Homer wrote the Odyssey."}], "style": "wiki"}
{"messages": [{"role": "system", "content": "style=books"}, {"role": "user", "content": "Recommend a fantasy series."}, {"role": "assistant", "content": "Try The Wheel of Time by Robert Jordan."}], "style": "books"}
{"messages": [{"role": "system", "content": "style=news"}, {"role": "user", "content": "Summarize today's headlines."}, {"role": "assistant", "content": "Major markets rallied; new policies were announced."}], "style": "news"}
{"messages": [{"role": "system", "content": "style=wiki"}, {"role": "user", "content": "What is photosynthesis?"}, {"role": "assistant", "content": "Photosynthesis converts light energy into chemical energy."}], "style": "wiki"}
EOF
fi

export JAX_PLATFORMS=${JAX_PLATFORMS:-cpu}

mkdir -p /content/data/style_demo/cache

python -m levanter.main.train_lm --config_path "$SANITIZED_CONFIG"

